# LoRA router: Qwen3-0.6B for the Nevados offline assistant

Trains a LoRA on Qwen3-0.6B for the FSM's `route` and `extract` calls, then exports a
`Q4_K_M` GGUF for llama.cpp (b12375, same as the phone).

**How to run**
1. Runtime → Change runtime type → **L4 GPU** (A100 also works).
2. Upload `nevados_ft.zip` (from `data/finetune/` in the repo) to Google Drive at `MyDrive/nevados-ft/`.
3. Edit `RUN_NAME` below if you want, then **Runtime → Run all**. About 45–60 min on an L4.
4. Results land in `MyDrive/nevados-ft/runs/<RUN_NAME>/`: the GGUF, the adapter and `metrics.json`.


In [ ]:
RUN_NAME = "r1"           # change per iteration
DRIVE_DIR = "/content/drive/MyDrive/nevados-ft"
BASE_MODEL = "Qwen/Qwen3-0.6B"
LLAMA_CPP_TAG = "b12375"  # same build as the phone and the benchmark
MAX_LEN = 3072
EPOCHS = 1
LR = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
BATCH, GRAD_ACC = 8, 2
SEED = 7
EVAL_N = 400              # dev examples for the greedy exact-match check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "transformers>=4.51,<5" "peft>=0.15" "accelerate>=1.6" datasets sentencepiece

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, json
os.makedirs("/content/data", exist_ok=True)
with zipfile.ZipFile(f"{DRIVE_DIR}/nevados_ft.zip") as z:
    z.extractall("/content/data")
print(os.listdir("/content/data"))
RUN_DIR = f"{DRIVE_DIR}/runs/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)

In [ ]:
import torch, random
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.manual_seed(SEED); random.seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, attn_implementation="sdpa").cuda()

def load(path):
    return [json.loads(l) for l in open(path) if l.strip()]

def encode(rec):
    p = tok(rec["prompt"], add_special_tokens=False)["input_ids"]
    c = tok(rec["completion"], add_special_tokens=False)["input_ids"]
    return {"input_ids": p + c, "labels": [-100] * len(p) + c}

train_raw, dev_raw = load("/content/data/train.jsonl"), load("/content/data/dev.jsonl")
train = [e for e in map(encode, train_raw) if len(e["input_ids"]) <= MAX_LEN]
dev = [e for e in map(encode, dev_raw) if len(e["input_ids"]) <= MAX_LEN]
lens = sorted(len(e["input_ids"]) for e in train)
print(f"train {len(train)}/{len(train_raw)}  dev {len(dev)}/{len(dev_raw)}  len p50 {lens[len(lens)//2]} max {lens[-1]}")
# The template must round-trip: special tokens are parsed, not split into characters.
assert tok.convert_ids_to_tokens(train[0]["input_ids"][:1]) == ["<|im_start|>"]

In [ ]:
from peft import LoraConfig, get_peft_model
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

In [ ]:
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    ids = torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch])
    lab = torch.tensor([b["labels"] + [-100] * (n - len(b["labels"])) for b in batch])
    att = torch.tensor([[1] * len(b["input_ids"]) + [0] * (n - len(b["input_ids"])) for b in batch])
    return {"input_ids": ids, "attention_mask": att, "labels": lab}

class CompletionOnlyTrainer(Trainer):
    # Logits only where labels exist: a 2k-token prompt x 151k vocab would not fit otherwise.
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        inner = model.base_model.model
        hidden = inner.model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).last_hidden_state
        h, tgt = hidden[:, :-1], labels[:, 1:]
        mask = tgt != -100
        logits = inner.lm_head(h[mask]).float()
        loss = F.cross_entropy(logits, tgt[mask])
        return (loss, None) if return_outputs else loss

args = TrainingArguments(
    output_dir="/content/out", num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine",
    warmup_ratio=0.03, per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, bf16=True, logging_steps=20, eval_strategy="no",
    save_strategy="no", report_to="none", seed=SEED, group_by_length=True,
    remove_unused_columns=False, prediction_loss_only=True, dataloader_num_workers=2)
trainer = CompletionOnlyTrainer(model=model, args=args, train_dataset=train, eval_dataset=dev, data_collator=collate)
train_result = trainer.train()
eval_result = {}  # dev quality is measured by the greedy exact-match cell below
print(train_result.metrics, eval_result)

In [ ]:
# Greedy exact match on dev (no grammar here; llama.cpp adds the JSON schema in deployment).
model.eval(); model.config.use_cache = True
tok.padding_side = "left"
sample = random.Random(SEED).sample(dev_raw, min(EVAL_N, len(dev_raw)))
hits, totals, misses = {}, {}, []
for i in range(0, len(sample), 16):
    chunk = sample[i:i + 16]
    enc = tok([r["prompt"] for r in chunk], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=96, do_sample=False, pad_token_id=tok.pad_token_id)
    for r, seq in zip(chunk, out[:, enc["input_ids"].shape[1]:]):
        text = tok.decode(seq, skip_special_tokens=True).strip()
        want = json.loads(r["completion"].replace("<|im_end|>", ""))
        try:
            got = json.loads(text)
        except json.JSONDecodeError:
            got = None
        key = r["call_type"]
        totals[key] = totals.get(key, 0) + 1
        ok = got == want
        hits[key] = hits.get(key, 0) + ok
        if not ok and len(misses) < 25:
            misses.append({"want": want, "got": text})
dev_exact = {k: round(hits[k] / totals[k], 4) for k in totals}
print("dev exact match:", dev_exact)
for m in misses[:10]: print(m)

In [ ]:
# Save adapter, merge, export GGUF f16 -> Q4_K_M with llama.cpp at the phone's build.
model.save_pretrained(f"{RUN_DIR}/adapter")
merged = model.merge_and_unload()
merged.save_pretrained("/content/merged", safe_serialization=True)
tok.padding_side = "right"
tok.save_pretrained("/content/merged")
!rm -rf /content/llama.cpp && git clone -q --depth 1 --branch {LLAMA_CPP_TAG} https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q /content/llama.cpp/gguf-py
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outtype f16 --outfile /content/router-f16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/router-f16.gguf /content/router-Q4_K_M.gguf Q4_K_M | tail -3

In [ ]:
import shutil, hashlib
gguf_name = f"qwen3-0.6b-router-{RUN_NAME}-Q4_K_M.gguf"
shutil.copy("/content/router-Q4_K_M.gguf", f"{RUN_DIR}/{gguf_name}")
sha = hashlib.sha256(open("/content/router-Q4_K_M.gguf", "rb").read()).hexdigest()
metrics = {"run": RUN_NAME, "base": BASE_MODEL, "epochs": EPOCHS, "lr": LR, "lora_r": LORA_R,
           "lora_alpha": LORA_ALPHA, "train_n": len(train), "dev_n": len(dev),
           "train": train_result.metrics, "eval": eval_result, "dev_exact": dev_exact,
           "misses": misses, "gguf": gguf_name, "sha256": sha}
json.dump(metrics, open(f"{RUN_DIR}/metrics.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps({k: metrics[k] for k in ("dev_exact", "gguf", "sha256")}, indent=2))
print("Done. Download", f"{RUN_DIR}/{gguf_name}", "and metrics.json")